# AI for Radiologic Image Tasks - *Mastering Artificial Intelligence in Radiology*
## Detection

This notebook introduces a complete, educational workflow for training a simple lung detector on chest X-ray images. The task is to predict two rectangular bounding boxes: one around the left lung and one around the right lung.

The notebook uses the NLM Montgomery dataset. Unlike the classification notebook, the original labels here are not image-level disease labels. They are manual lung masks, which mark lung pixels directly. We convert those masks into bounding boxes so the problem becomes a lightweight object-detection task that can be trained with a ResNet-34 box-regression model.

The main steps are:

1. Build an image index with image paths, mask paths, and readable disease labels.
2. Convert left-lung and right-lung masks into bounding-box coordinates.
3. Create a random train/validation split from the Montgomery images.
4. Prepare PyTorch datasets and loaders, including box-aware image augmentation.
5. Define a ResNet-34 model that regresses two lung boxes per image.
6. Train the model while tracking box-regression loss and intersection-over-union.
7. Preview ground-truth boxes beside model predictions on validation images.

The goal is not to build a production-ready detector. Instead, the notebook is meant to show the core logic behind detection: turn annotations into spatial targets, keep images and boxes aligned during preprocessing, train a model to predict coordinates, and evaluate whether the predicted boxes overlap the expected anatomy.


## 0. Setup

This section imports the libraries, fixes the random seed for repeatable results, defines the Montgomery dataset folders, and selects the compute device.

Before running the notebook, replace `DATASET_DIR` in the code cell below with the location of the Montgomery dataset on your own environment. The notebook expects the dataset directory to contain a `CXR_png` folder and the two manual-mask folders `ManualMask/leftMask` and `ManualMask/rightMask`.


In [ ]:
from pathlib import Path
import random
import warnings

import albumentations as A
import cv2
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet34
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["figure.figsize"] = (8, 8)
plt.rcParams["image.cmap"] = "gray"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATASET_DIR = Path("/media/felipe/KINGSTON/datasets/NLM-MontgomeryCXRSet/MontgomerySet")
IMAGE_DIR = DATASET_DIR / "CXR_png"
LEFT_MASK_DIR = DATASET_DIR / "ManualMask" / "leftMask"
RIGHT_MASK_DIR = DATASET_DIR / "ManualMask" / "rightMask"
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

assert IMAGE_DIR.exists(), f"Image directory not found: {IMAGE_DIR}"
assert LEFT_MASK_DIR.exists(), f"Left-mask directory not found: {LEFT_MASK_DIR}"
assert RIGHT_MASK_DIR.exists(), f"Right-mask directory not found: {RIGHT_MASK_DIR}"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute device selected for this notebook: {DEVICE}")
print(f"PyTorch runtime version and CUDA build: torch {torch.__version__}; cuda {torch.version.cuda}")


## 1. Build Image And Mask Index

Here we create a simple table with one row per image. Each row stores the image ID, image path, left-mask path, right-mask path, numeric disease label, and readable label name. This table becomes the source of truth for both the image pixels and the spatial annotations used later in the notebook.

The disease label is not used as the detection target. It is kept only as descriptive metadata, while the detection model learns lung-box coordinates from the mask-derived annotations.


In [ ]:
def disease_label_from_id(image_id: str) -> int:
    return int(image_id.rsplit("_", 1)[-1])


def build_montgomery_index() -> pd.DataFrame:
    rows = []
    for image_path in sorted(IMAGE_DIR.glob("*.png")):
        left_mask_path = LEFT_MASK_DIR / image_path.name
        right_mask_path = RIGHT_MASK_DIR / image_path.name
        if not left_mask_path.exists() or not right_mask_path.exists():
            continue
        image_id = image_path.stem
        rows.append({
            "ImageId": image_id,
            "image_path": image_path,
            "left_mask_path": left_mask_path,
            "right_mask_path": right_mask_path,
            "disease_label": disease_label_from_id(image_id),
        })
    return pd.DataFrame(rows)


images_df = build_montgomery_index()
assert not images_df.empty, "No complete image/left-mask/right-mask triplets found."
print(f"Complete image/mask pairs found for detection: {len(images_df):,}")
print("Disease-label counts kept only as descriptive metadata:")
images_df["disease_label"].value_counts().rename({0: "normal", 1: "abnormal"})

## 2. Convert Masks To Boxes

The Montgomery annotations are manual segmentation masks: each mask identifies the pixels that belong to one lung. A bounding box is a simpler detection label that stores only four numbers: **x_min**, **y_min**, **x_max**, and **y_max**.

This section reads each left-lung and right-lung mask, finds the outermost nonzero mask pixels, and converts that pixel region into one rectangular box. Each image should produce two boxes, one for each lung.


In [ ]:
def read_grayscale_png(path: Path) -> np.ndarray:
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(path)
    return image


def normalize_percentile(image: np.ndarray, lower: float = 1, upper: float = 99, eps: float = 1e-7) -> np.ndarray:
    image = image.astype(np.float32)
    lo, hi = np.percentile(image, [lower, upper])
    image = np.clip(image, lo, hi)
    return ((image - lo) / (hi - lo + eps)).astype(np.float32)


def mask_to_box(mask: np.ndarray) -> list[float] | None:
    ys, xs = np.where(mask > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None
    x_min = float(xs.min())
    y_min = float(ys.min())
    x_max = float(xs.max() + 1)
    y_max = float(ys.max() + 1)
    return [x_min, y_min, x_max, y_max]


def lung_boxes(row: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    boxes = []
    labels = []
    for mask_path in [row.left_mask_path, row.right_mask_path]:
        box = mask_to_box(read_grayscale_png(mask_path))
        if box is not None:
            boxes.append(box)
            labels.append(1)
    return np.array(boxes, dtype=np.float32), np.array(labels, dtype=np.int64)


def image_and_boxes(image_id: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    row = images_df.loc[images_df["ImageId"] == image_id].iloc[0]
    image = read_grayscale_png(row.image_path)
    boxes, labels = lung_boxes(row)
    return image, boxes, labels

The next cell applies the mask-to-box conversion to every indexed image and stores the resulting boxes in a dataframe. The assertion confirms that every image has at least one detected box before training continues.


In [ ]:
box_rows = []
for row in images_df.itertuples(index=False):
    boxes, labels = lung_boxes(pd.Series(row._asdict()))
    box_rows.append({
        "ImageId": row.ImageId,
        "boxes": boxes,
        "labels": labels,
        "n_boxes": len(boxes),
    })

boxes_df = images_df.merge(pd.DataFrame(box_rows), on="ImageId")
assert (boxes_df["n_boxes"] > 0).all(), "At least one image has no boxes."
print("First mask-derived box records; n_boxes should be 2 for paired lung boxes:")
boxes_df[["ImageId", "n_boxes", "disease_label"]].head()

The next cell displays one Montgomery image with its converted lung boxes. This is a quick sanity check that the mask-derived rectangles are aligned with the visible lung fields.


In [ ]:
def draw_boxes(ax, boxes, color="lime", linewidth=2):
    for box in boxes:
        x_min, y_min, x_max, y_max = box
        rect = patches.Rectangle(
            (x_min, y_min),
            x_max - x_min,
            y_max - y_min,
            linewidth=linewidth,
            edgecolor=color,
            facecolor="none",
        )
        ax.add_patch(rect)


sample_row = boxes_df.sample(1, random_state=SEED).iloc[0]
sample_image, sample_boxes, sample_labels = image_and_boxes(sample_row.ImageId)

print(f"Random sample selected for box-overlay sanity check: {sample_row.ImageId}")
print(f"Original grayscale image shape before resizing: {sample_image.shape}")
print(f"Mask-derived lung boxes in original pixel coordinates:\n{sample_boxes}")

fig, ax = plt.subplots(1, 1, figsize=(7, 7))
ax.imshow(normalize_percentile(sample_image), cmap="gray")
draw_boxes(ax, sample_boxes)
ax.set_title("Mask-derived lung boxes")
ax.axis("off")
plt.tight_layout()

## 3. Train/Validation Split

The Montgomery dataset is divided into two internal groups before training:

- **Training split:** 80% of the images, used to update the model weights.
- **Validation split:** 20% of the images, kept separate and used to monitor performance during training.

This validation split gives us a held-out check on the same dataset source. It helps answer whether the detector learned a reusable lung-localization pattern instead of memorizing the exact training images.


In [ ]:
def random_image_split(df: pd.DataFrame, valid_fraction: float = 0.2, seed: int = SEED):
    rng = np.random.default_rng(seed)
    ids = df["ImageId"].to_numpy().copy()
    rng.shuffle(ids)
    n_valid = max(1, int(round(len(ids) * valid_fraction)))
    valid_ids = ids[:n_valid]
    train_ids = ids[n_valid:]
    return train_ids, valid_ids


train_ids, valid_ids = random_image_split(boxes_df)
print("Random train/validation split size summary:")
pd.DataFrame({
    "split": ["all", "train", "valid"],
    "images": [len(boxes_df), len(train_ids), len(valid_ids)],
})


## 4. PyTorch Detection Dataset

This section prepares images and boxes for the model by defining the transformations, a custom PyTorch dataset class, and a detection-specific batch collation function.

Data augmentation is more delicate for detection than for classification. If an image is shifted, scaled, rotated, or flipped, the bounding boxes must be transformed in the same way. Otherwise, the image and label would no longer describe the same anatomy.

In this notebook, the training images use these augmentation techniques:

1. Resize: makes every image the same input size for the neural network.
2. Shift, scale, and rotation: slightly moves, zooms, or rotates the image and boxes together.
3. Horizontal flip: mirrors the image and updates the box coordinates.

Validation images are not randomly augmented. They are only resized, so validation loss and IoU are consistent and easier to interpret.


In [ ]:
def make_train_transform(size: int) -> A.Compose:
    return A.Compose(
        [
            A.Resize(height=size, width=size),
            A.ShiftScaleRotate(
                shift_limit=0.03,
                scale_limit=0.08,
                rotate_limit=7,
                p=0.5,
                border_mode=cv2.BORDER_CONSTANT,
            ),
            A.HorizontalFlip(p=0.5),
        ],
        bbox_params=A.BboxParams(format="pascal_voc", label_fields=["labels"], min_visibility=0.0),
    )


def make_valid_transform(size: int) -> A.Compose:
    return A.Compose(
        [A.Resize(height=size, width=size)],
        bbox_params=A.BboxParams(format="pascal_voc", label_fields=["labels"]),
    )


def sort_boxes_left_to_right(boxes: np.ndarray, labels: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    order = np.argsort(boxes[:, 0])
    return boxes[order], labels[order]


def collate_detection_batch(batch):
    images, targets, image_ids = zip(*batch)
    return list(images), list(targets), list(image_ids)


class MontgomeryLungDetectionDataset(Dataset):
    def __init__(self, image_ids, metadata: pd.DataFrame, transform: A.Compose | None = None):
        self.image_ids = list(image_ids)
        self.metadata = metadata.set_index("ImageId")
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        row = self.metadata.loc[image_id]
        image = normalize_percentile(read_grayscale_png(row.image_path))
        boxes, labels = lung_boxes(row)

        if self.transform is not None:
            transformed = self.transform(image=image, bboxes=boxes.tolist(), labels=labels.tolist())
            image = transformed["image"].astype(np.float32)
            boxes = np.array(transformed["bboxes"], dtype=np.float32)
            labels = np.array(transformed["labels"], dtype=np.int64)

        if len(boxes) != 2:
            raise ValueError(f"Expected two lung boxes for {image_id}, got {len(boxes)}")
        boxes, labels = sort_boxes_left_to_right(boxes, labels)

        image = np.repeat(image[..., None], 3, axis=-1)
        image = torch.from_numpy(np.ascontiguousarray(image.transpose(2, 0, 1))).float()

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": area,
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64),
        }
        return image, target, image_id


In the next cell, we use the dataset class defined above and choose the main data-loading settings for this project.

`IMAGE_SIZE` defines the standard image size used by the model. `BATCH_SIZE` controls how many images are processed at once. `NUM_WORKERS` controls how many background processes help load the images. The custom collate function keeps each image paired with its own target dictionary.


In [ ]:
IMAGE_SIZE = 512
BATCH_SIZE = 4
NUM_WORKERS = 4

# Keep these non-None for quick CPU smoke tests. Set to None for full training.
MAX_TRAIN_IMAGES = None
MAX_VALID_IMAGES = None

train_transform = make_train_transform(IMAGE_SIZE)
valid_transform = make_valid_transform(IMAGE_SIZE)

train_ids_run = train_ids[:MAX_TRAIN_IMAGES] if MAX_TRAIN_IMAGES else train_ids
valid_ids_run = valid_ids[:MAX_VALID_IMAGES] if MAX_VALID_IMAGES else valid_ids

train_ds = MontgomeryLungDetectionDataset(train_ids_run, boxes_df, transform=train_transform)
valid_ds = MontgomeryLungDetectionDataset(valid_ids_run, boxes_df, transform=valid_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_detection_batch,
)
valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_detection_batch,
)

images, targets, image_ids = next(iter(train_loader))
print("Training-loader smoke test: first image tensor shape, first target boxes, and first two image IDs.")
{
    "first_image_tensor_shape": tuple(images[0].shape),
    "first_target_boxes": targets[0]["boxes"],
    "first_two_image_ids": image_ids[:2],
}

The next cell shows how box-aware augmentation changes a training image. The first panel displays the resized image without random augmentation, and the other panels show different augmented versions created from the same original image with boxes transformed alongside it.


In [ ]:
preview_id = sample_row.ImageId
preview_image, preview_boxes, preview_labels = image_and_boxes(preview_id)
preview_image = normalize_percentile(preview_image).astype(np.float32)

base = valid_transform(image=preview_image, bboxes=preview_boxes.tolist(), labels=preview_labels.tolist())

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for col in range(4):
    if col == 0:
        transformed = base
        title = "Original"
    else:
        transformed = train_transform(image=preview_image, bboxes=preview_boxes.tolist(), labels=preview_labels.tolist())
        title = f"Albumentations {col}"
    axes[col].imshow(transformed["image"], cmap="gray", vmin=0, vmax=1)
    draw_boxes(axes[col], transformed["bboxes"])
    axes[col].set_title(title)
    axes[col].axis("off")
plt.tight_layout()

## 5. Teaching Point

Detection changes what the model has to learn. In classification, one image has one target such as `TB` or `No TB`. In this notebook, one image has spatial targets: two rectangles with coordinates.

### The label must move with the image.

A horizontal flip is harmless for classification because the disease label stays the same. For detection, the box coordinates must also flip. The same idea applies to resizing, shifting, scaling, and rotation. Good detection preprocessing always treats the image and annotation as a paired object.

This is why the transform pipeline uses `bbox_params` and why the dataset checks that exactly two boxes remain after augmentation. If an augmentation drops or corrupts a box, the model would receive a misleading training example.

**Takeaway:** object detection is not only about predicting where something is. It is also about preserving the geometric relationship between the image and its labels at every preprocessing step.


## 6. ResNet-34 Box Regressor

This section defines a ResNet-34 model for bounding-box regression. The model receives a chest X-ray image and outputs eight numbers: four coordinates for the left lung box and four coordinates for the right lung box.

The network predicts normalized coordinates between 0 and 1 by using a sigmoid output. During training, the ground-truth boxes are also normalized by image size. During visualization, the predicted boxes are converted back into pixel coordinates so they can be drawn on the image.

The original ResNet-34 classifier is replaced with a small regression head. This keeps the notebook focused on fixed lung-box prediction while using the same ResNet-34 backbone style as the other model notebooks.


In [ ]:
class LungBoxRegressor(nn.Module):
    def __init__(self, num_boxes: int = 2):
        super().__init__()
        self.num_boxes = num_boxes
        self.model = resnet34(weights=None)
        in_features = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(in_features, num_boxes * 4),
        )

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        return torch.sigmoid(self.model(images)).view(-1, self.num_boxes, 4)


def normalize_boxes(boxes: torch.Tensor, image_size: int = IMAGE_SIZE) -> torch.Tensor:
    return boxes / float(image_size)


def denormalize_boxes(boxes: torch.Tensor, image_size: int = IMAGE_SIZE) -> torch.Tensor:
    return boxes * float(image_size)


def ordered_boxes(boxes: torch.Tensor) -> torch.Tensor:
    top_left = torch.minimum(boxes[..., :2], boxes[..., 2:])
    bottom_right = torch.maximum(boxes[..., :2], boxes[..., 2:])
    return torch.cat([top_left, bottom_right], dim=-1)


model = LungBoxRegressor(num_boxes=2).to(DEVICE)
model.train()
with torch.no_grad():
    smoke_predictions = model(torch.stack([image.to(DEVICE) for image in images[:2]]))
print("Model smoke test output shape: batch size, number of boxes, and four coordinates per box.")
smoke_predictions.shape


## 7. Train

Training repeatedly shows image batches to the model, compares predicted boxes with ground-truth boxes, and updates the model weights. The notebook tracks two quantities:

- **Smooth L1 loss:** the coordinate-regression loss used for optimization.
- **Intersection-over-union (IoU):** an overlap score between predicted and target boxes.

Higher IoU means the predicted boxes overlap the manual-mask-derived boxes more closely. The best checkpoint is selected by validation loss, while IoU gives a more interpretable sense of localization quality.


In [ ]:
def stack_images(images, device):
    return torch.stack([image.to(device, non_blocking=True) for image in images])


def target_boxes_to_tensor(targets, device, image_size: int = IMAGE_SIZE):
    boxes = torch.stack([normalize_boxes(target["boxes"], image_size) for target in targets])
    return boxes.to(device, non_blocking=True)


def box_regression_loss(pred_boxes: torch.Tensor, target_boxes: torch.Tensor) -> torch.Tensor:
    return F.smooth_l1_loss(pred_boxes, target_boxes)


def aligned_box_iou(pred_boxes: torch.Tensor, target_boxes: torch.Tensor, eps: float = 1e-7) -> torch.Tensor:
    pred_boxes = ordered_boxes(pred_boxes)
    target_boxes = ordered_boxes(target_boxes)

    inter_top_left = torch.maximum(pred_boxes[..., :2], target_boxes[..., :2])
    inter_bottom_right = torch.minimum(pred_boxes[..., 2:], target_boxes[..., 2:])
    inter_size = (inter_bottom_right - inter_top_left).clamp(min=0)
    intersection = inter_size[..., 0] * inter_size[..., 1]

    pred_size = (pred_boxes[..., 2:] - pred_boxes[..., :2]).clamp(min=0)
    target_size = (target_boxes[..., 2:] - target_boxes[..., :2]).clamp(min=0)
    pred_area = pred_size[..., 0] * pred_size[..., 1]
    target_area = target_size[..., 0] * target_size[..., 1]
    union = pred_area + target_area - intersection
    return ((intersection + eps) / (union + eps)).mean()


def run_train_epoch(model, loader, optimizer):
    model.train()
    totals = {"loss": 0.0, "iou": 0.0}
    total_batches = 0

    for images, targets, _ in tqdm(loader, leave=False):
        images = stack_images(images, DEVICE)
        target_boxes = target_boxes_to_tensor(targets, DEVICE)

        pred_boxes = model(images)
        loss = box_regression_loss(pred_boxes, target_boxes)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        totals["loss"] += loss.item()
        totals["iou"] += aligned_box_iou(pred_boxes.detach(), target_boxes).item()
        total_batches += 1

    return {key: value / total_batches for key, value in totals.items()}


@torch.no_grad()
def run_valid_loss_epoch(model, loader):
    model.eval()
    totals = {"loss": 0.0, "iou": 0.0}
    total_batches = 0

    for images, targets, _ in tqdm(loader, leave=False):
        images = stack_images(images, DEVICE)
        target_boxes = target_boxes_to_tensor(targets, DEVICE)
        pred_boxes = model(images)
        loss = box_regression_loss(pred_boxes, target_boxes)

        totals["loss"] += loss.item()
        totals["iou"] += aligned_box_iou(pred_boxes, target_boxes).item()
        total_batches += 1

    return {key: value / total_batches for key, value in totals.items()}


The next cell is the actual training loop. It sets the number of epochs, learning-rate schedule, checkpoint path, and optimizer. Each epoch trains on the Montgomery training split, evaluates on the validation split, records loss and IoU, saves the best checkpoint by validation loss, advances the scheduler, and prints a compact progress summary.


In [ ]:
EPOCHS = 25
LEARNING_RATE = 1e-3
MIN_LEARNING_RATE = 1e-5
CHECKPOINT_PATH = RESULTS_DIR / "resnet34_box_regressor_montgomery_lung_boxes.pt"

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LEARNING_RATE)
history = []
best_valid_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_train_epoch(model, train_loader, optimizer)
    valid_metrics = run_valid_loss_epoch(model, valid_loader)
    current_lr = optimizer.param_groups[0]["lr"]
    history.append({"epoch": epoch, "lr": current_lr, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"valid_{k}": v for k, v in valid_metrics.items()}})

    if valid_metrics["loss"] < best_valid_loss:
        best_valid_loss = valid_metrics["loss"]
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "history": history,
            "image_size": IMAGE_SIZE,
        }, CHECKPOINT_PATH)

    scheduler.step()

    print(
        f"Training progress after epoch {epoch:02d}: lr {current_lr:.2e} | "
        f"train loss {train_metrics['loss']:.4f} iou {train_metrics['iou']:.4f} | "
        f"valid loss {valid_metrics['loss']:.4f} iou {valid_metrics['iou']:.4f}"
    )
print("Training history table recorded across epochs:")
pd.DataFrame(history)


## 8. Validate And Preview Predictions

After training, the trained model is used to predict lung boxes on held-out validation images. The preview plots each image twice: the left panel shows the ground-truth boxes derived from manual masks, and the right panel shows the model predictions.

This visual check is important because coordinate losses and IoU scores do not show the full failure mode. A detector can have a reasonable average score while still making clinically obvious localization errors on individual images. Looking at examples helps connect the metric back to the anatomy.


In [ ]:
@torch.no_grad()
def predict_batch(model, images):
    model.eval()
    image_batch = stack_images(images, DEVICE)
    pred_boxes = denormalize_boxes(ordered_boxes(model(image_batch))).cpu().numpy()
    return [
        {
            "boxes": boxes,
            "scores": np.ones((boxes.shape[0],), dtype=np.float32),
            "labels": np.ones((boxes.shape[0],), dtype=np.int64),
        }
        for boxes in pred_boxes
    ]


def tensor_to_display_image(image: torch.Tensor) -> np.ndarray:
    return np.clip(image.detach().cpu().numpy().transpose(1, 2, 0)[..., 0], 0, 1)


preview_valid_ids = valid_ids[:max(BATCH_SIZE, 4)]
preview_valid_ds = MontgomeryLungDetectionDataset(preview_valid_ids, boxes_df, transform=valid_transform)
preview_valid_loader = DataLoader(
    preview_valid_ds,
    batch_size=min(BATCH_SIZE, len(preview_valid_ds)),
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_detection_batch,
)

images, targets, image_ids = next(iter(preview_valid_loader))
predictions = predict_batch(model, images)

n = min(4, len(images))
fig, axes = plt.subplots(n, 2, figsize=(10, 4 * n))
if n == 1:
    axes = axes[None, :]

for i in range(n):
    display_image = tensor_to_display_image(images[i])
    axes[i, 0].imshow(display_image, cmap="gray", vmin=0, vmax=1)
    draw_boxes(axes[i, 0], targets[i]["boxes"].numpy(), color="lime")
    axes[i, 0].set_title(f"{image_ids[i]} ground truth")

    axes[i, 1].imshow(display_image, cmap="gray", vmin=0, vmax=1)
    draw_boxes(axes[i, 1], predictions[i]["boxes"], color="red")
    axes[i, 1].set_title(f"prediction ({len(predictions[i]['boxes'])} boxes)")

    for ax in axes[i]:
        ax.axis("off")
plt.tight_layout()


The final cell plots the training history. The loss curve shows whether optimization improved over time, and the learning-rate plot shows the cosine schedule used during training.


In [ ]:
history_df = pd.DataFrame(history)
loss_columns = [column for column in history_df.columns if column.endswith("loss") or column in {"train_loss", "valid_loss"}]
history_df.plot(x="epoch", y=sorted(set(loss_columns)), figsize=(8, 4))
plt.grid(True, alpha=0.3)
plt.tight_layout()

history_df.plot(x="epoch", y="lr", figsize=(8, 2), logy=True, legend=False)
plt.ylabel("learning rate")
plt.grid(True, alpha=0.3)
plt.tight_layout()